# 4. Reinforcement-learning fine-tuning

Generating a list is an episode of $K$ steps with state $s_k = (h, a_{1:k-1})$ and action
$a_k \sim \pi_\theta(\cdot \mid s_k)$. The reward is the response $R_{\text{train}}$ predicts to $a_k$; for the items
generated before $a_k$ the reward model is given its own expected played ratio as their feedback:

$$r_k = \hat\rho\,(a_k \mid h, a_{1:k-1}), \qquad G_k = \sum_{l \ge k} \gamma^{\,l-k}\, r_l .$$

A value model $V_\phi$, initialised from the recommender, regresses on $G_k$ and yields the advantages by
generalised advantage estimation,

$$\delta_k = r_k + \gamma V_\phi(s_{k+1}) - V_\phi(s_k), \qquad A_k = \sum_{l \ge 0} (\gamma\lambda)^l\, \delta_{k+l},$$

which are standardised within each batch. The policy minimises the clipped surrogate of PPO with a KL penalty
towards the base recommender $\pi_0$:

$$\mathcal{L}(\theta) = -\,\mathbb{E}\Big[\min\big(w_k A_k,\ \operatorname{clip}(w_k, 1 - \epsilon, 1 + \epsilon)\, A_k\big)\Big]
+ \beta\, \mathbb{E}\Big[\operatorname{KL}\big(\pi_\theta(\cdot \mid s_k)\,\|\,\pi_0(\cdot \mid s_k)\big)\Big],
\qquad w_k = \frac{\pi_\theta(a_k \mid s_k)}{\pi_{\theta_{\text{old}}}(a_k \mid s_k)} .$$

The reward contains no relevance term, so $\beta$ is the only constraint that ties the policy to observed
behaviour.

In [ ]:
import numpy as np
import pandas as pd

from gptrec.data.batches import build_eval_examples
from gptrec.data.log import InteractionLog
from gptrec.evaluation.behavior import list_overlap
from gptrec.evaluation.policy_report import EvaluationConfig, evaluate_policy, metrics_table, per_step_table
from gptrec.paths import ARTIFACTS_DIR, PROCESSED_DATA_DIR
from gptrec.rollout import GenerationConfig, RewardSpec, SimulatedFeedback
from gptrec.training.checkpoint import load_policy, load_reward_model, save_model
from gptrec.training.precision import default_device
from gptrec.training.rl import RLConfig, finetune_with_rl

device = default_device()
log = InteractionLog.load(PROCESSED_DATA_DIR / "yambda-50m")

In [ ]:
pretrained = load_policy(ARTIFACTS_DIR / "policy_pretrained.pt", device)
reward_models = {
    "train": load_reward_model(ARTIFACTS_DIR / "reward_model_a.pt", device),
    "heldout": load_reward_model(ARTIFACTS_DIR / "reward_model_b.pt", device),
}

HORIZON = 10
HISTORY_LEN = pretrained.config.transformer.max_positions - HORIZON
MAX_FUTURE = 20

evaluation_config = EvaluationConfig(ks=(1, 5, 10), generation=GenerationConfig(horizon=HORIZON, greedy=True))
validation_examples = build_eval_examples(log, "validation", HISTORY_LEN, MAX_FUTURE)
test_examples = build_eval_examples(log, "test", HISTORY_LEN, MAX_FUTURE)


def evaluate(policy, examples):
    return evaluate_policy(policy, examples, log.items, reward_models, evaluation_config, device)

## Training

In [ ]:
MONITORED = [
    "top_k/all/ndcg@10",
    "next_k/all/ndcg@10",
    "next_k/liked/ndcg@10",
    "predicted/train/played_ratio",
    "predicted/heldout/played_ratio",
    "next_k/profile/catalog_coverage",
]


def monitor(policy) -> dict[str, float]:
    metrics = evaluate(policy, validation_examples).metrics
    return {name: metrics[name] for name in MONITORED}


rl_config = RLConfig(
    num_steps=2000,
    histories_per_step=512,
    ppo_epochs=4,
    history_len=HISTORY_LEN,
    max_future=MAX_FUTURE,
    generation=GenerationConfig(horizon=HORIZON),
    simulated_feedback=SimulatedFeedback.EXPECTED,
    reward=RewardSpec(played_ratio_weight=1.0),
    reference_kl_weight=0.3,
    learning_rate=1e-5,
    evaluate_every=200,
)
rl_policy, records = finetune_with_rl(pretrained, reward_models["train"], log, rl_config, device, evaluate=monitor)
save_model(rl_policy, ARTIFACTS_DIR / "policy_rl.pt")
records = pd.DataFrame(records).set_index("step")
records.to_csv(ARTIFACTS_DIR / "policy_rl_training.csv")

Per iteration, over the sampled lists: mean reward, $\operatorname{KL}(\pi_\theta \,\|\, \pi_0)$, entropy, the share
of distinct items, the share of items in $F$, the share of items from the history, mean item popularity and
duration, the gradient norm, the fraction of clipped ratios, and the variance of $G$ explained by $V_\phi$. Below:
relevance, predicted played ratio and catalogue coverage of greedy lists for validation users.

In [ ]:
training_curves = ["reward", "kl_to_reference", "entropy", "distinct_action_share", "logged_hit_rate", "history_repeat_rate", "mean_action_popularity", "mean_action_duration_seconds", "gradient_norm", "clip_fraction", "value_explained_variance"]
records[training_curves].rolling(20, min_periods=1).mean().plot(subplots=True, layout=(2, 6), figsize=(30, 6));

In [ ]:
validation_curves = records[[f"validation/{name}" for name in MONITORED]].dropna()
validation_curves.plot(subplots=True, layout=(2, 3), figsize=(18, 6), marker="o");

## Test period

Fine-tuned lists are compared with those of $\pi_0$ by relevance, by the response both reward models predict, and
by concentration: catalogue coverage, the Gini coefficient of item exposure, the share of users who receive the
most frequent item, and the expected overlap between the lists of two users.

In [ ]:
evaluations = {"pretrained": evaluate(pretrained, test_examples), "rl": evaluate(rl_policy, test_examples)}
comparison = metrics_table(evaluations)
comparison["relative change"] = comparison["rl"] / comparison["pretrained"] - 1
with pd.option_context("display.max_rows", None, "display.float_format", "{:.4f}".format):
    display(comparison)

Share of the items of a fine-tuned list that the list of $\pi_0$ for the same user also contains:

In [ ]:
pd.Series({
    "next-k": list_overlap(evaluations["rl"].generated.actions, evaluations["pretrained"].generated.actions),
    "top-k": list_overlap(evaluations["rl"].top_k, evaluations["pretrained"].top_k),
})

## Exposure

The exposure of an item is the number of test users whose list contains it. The items with the largest gain and
the largest loss from $\pi_0$ to $\pi_\theta$, and the rank correlation of the change with item popularity,
duration and mean played ratio:

In [ ]:
def exposure(evaluation) -> np.ndarray:
    return np.bincount(evaluation.generated.actions.flatten().numpy(), minlength=log.num_items + 1)


items = pd.DataFrame({
    "popularity": log.items.train_popularity,
    "duration_seconds": log.items.duration_seconds,
    "mean_played_ratio": log.items.mean_played_ratio,
    "exposure_pretrained": exposure(evaluations["pretrained"]),
    "exposure_rl": exposure(evaluations["rl"]),
}).iloc[1:].rename_axis("item token")
items["exposure_change"] = items.exposure_rl - items.exposure_pretrained

display(items.sort_values("exposure_change", ascending=False).head(20))
display(items.sort_values("exposure_change").head(20))
items[["popularity", "duration_seconds", "mean_played_ratio"]].rank().corrwith(items.exposure_change.rank())

## List positions

In [ ]:
for name, evaluation in evaluations.items():
    print(name)
    display(per_step_table(evaluation, test_examples, log.items))